The emotion word masking list and processing pipeline are presented below, including emotion labels as well as their synonyms and morphological variants.

In [ ]:
# -*- coding: utf-8 -*-
import os
import re
import pandas as pd

# ========== 1) Emotion Word List ==========
EMO_PATTERN = re.compile(
    r"\b("
    r"angry|anger|furious|mad|irritated?|annoyed?|frustrat(ed|ing|ion)?|"
    r"sad|sadness|unhappy|depress(ed|ing|ion)?|upset|down|miserable|cry(ing)?|"
    r"fear|afraid|scare(d|y)|terrified|anxious|anxiety|nervous|panic|"
    r"disgust(ed|ing)?|gross|repuls(ed|ive)|nauseat(ed|ing)?|"
    r"surprise(d)?|shocked|astonish(ed|ing)|amaze(d|ment)|startl(ed|ing)?|"
    r"happy|happiness|glad|joy(ful)?|delight(ed|ful)?|pleased|cheerful|excited|thrill(ed|ing)?|"
    r"neutral|calm|relax(ed)?|content|fine|okay|"
    r"positive|negative"
    r")\b",
    re.IGNORECASE
)

def mask_emo_words(text: str, mask_token: str = "<mask>") -> str:
    """Replace matched emotion words with mask_token."""
    if text is None:
        return ""
    s = str(text)
    return EMO_PATTERN.sub(mask_token, s)

def main():
   # ========== 2) Path Configuration ==========
    IN_CSV  = "/root/autodl-tmp/test_Dia_labeled.csv"
    OUT_CSV = "/root/autodl-tmp/MELD/Test_Dia_labeled_semantic_masked.csv"

   # ========== 3) I/O Operations ==========
    df = pd.read_csv(IN_CSV, encoding="utf-8-sig")

    col = "semantic_explanation"
    if col not in df.columns:
        raise RuntimeError(f"Column '{col}' not found in CSV. Available cols: {list(df.columns)[:30]} ...")

    # Add a new column (keep the original column unchanged)
    new_col = f"{col}__masked"
    df[new_col] = df[col].apply(lambda x: mask_emo_words(x, mask_token="<mask>"))

    # Optional: count the number of processed lines
    hit_ratio = (df[new_col] != df[col].fillna("").astype(str)).mean()
    print(f"Masked rows ratio: {float(hit_ratio):.4f}")

    # Save
    os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
    df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print("✅ Done")
    print("Input :", IN_CSV)
    print("Output:", OUT_CSV)
    print("Rows  :", len(df))
    print("New col:", new_col)

if __name__ == "__main__":
    main()
